# Production plan with capacity options

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/capacity_options.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the fifth version of the model, M4, the orders come from six retailers at the end of a distribution network in Spain, and part of the demand that a retailer does not receive in its month is lost. This notebook uses M4B, a version built on M4 for a demand that is not the one forecast. The orders of each month can grow over the base forecast, the price of a month can fall with a promotion, and the plan has three capacity options: overtime at plants 1 and 2, rented storage next to the central warehouse, and plant 3 open or closed with its fixed cost.

Contents of the notebook:

1. Base forecast, orders of each retailer and capacity options.
2. Linear model with capacity options.
3. Plan of a scenario, built with the form: demand growth, price promotion, plant 3, overtime and rented storage.

The cells run in order, with ▶ or *Shift + Enter*. The form of the last cell (sliders, tick boxes and a drop-down list) builds the scenario: after changing a value, run the cell again. Outside Colab the form is a set of lines such as `demand_growth = 0`: edit the value and run the cell.

The table gives the symbols that M4B adds to those of M0 to M4, which the [document of M4](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/retailers.md) lists.

| Symbol | Meaning | Term | Unit |
|---|---|---|---|
| $D_t$ | Orders of the six retailers together in month $t$ before any growth | base forecast | bikes/month |
| $\sigma_r$ | Share of the orders of each month that retailer $r$ places | — | — |
| $\gamma_t$ | Rise of the orders of every retailer in month $t$ over the base forecast | demand growth | — |
| $\pi_t$ | Price that a retailer pays for a bike ordered in month $t$, which a price promotion lowers by its discount; it is also the lost-sale cost of that month | price; discount | €/bike |
| $\bar o_f$ | Bikes that plant $f$ can assemble in a month above its assembly capacity | overtime | bikes/month |
| $c^o$ | Cost of a bike assembled in overtime above the unit assembly cost of its plant | — | €/bike |
| $\bar u$ | Bikes that can be held next to the central warehouse above its storage capacity | rented storage | bikes |
| $c^u$ | Cost of holding one bike in rented storage for one month, above the holding cost of the central warehouse | — | €/(bike·month) |
| $c^F_f$ | Cost of keeping plant $f$ open for one month, whatever it assembles | fixed cost | €/month |
| $o_{ft}$ | Bikes that plant $f$ assembles in overtime in month $t$, of the $p_{ft}$ it assembles | overtime | bikes/month |
| $u_t$ | Bikes held in rented storage at the end of month $t$, of the $h_{wt}$ held at the central warehouse | rented storage | bikes |
| $\delta_f$ | 1 if plant $f$ is open from January to June, 0 if it is closed | — | binary |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) explains all the data and results. The places and distances are real; the other data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import capacity_options as co

## 1. Base forecast, orders of each retailer and capacity options

The first table gives the base forecast, the orders of the six retailers together in each month, the orders of each retailer, which is its share of it, and the price of a bike in each month. The second gives the capacity options and their cost. The rest of the data are those of M4.

In [ ]:
# @title Base forecast, orders and capacity options { display-mode: "form" }
co.show_data()

## 2. Linear model with capacity options

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} \left( c_f\, p_{ft} + c^o\, o_{ft} \right) + \sum_{i \in V} c^h_i\, h_{it} + c^u\, u_t + \sum_{a \in A} c^x_a\, x_{at} + \sum_{r \in R} \left( c^b_r\, b_{rt} + \pi_t\, l_{rt} \right) \Big) + \sum_{f \in F} |T|\, c^F_f\, \delta_f \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{r,t-1} + d_{rt} = s_{rt} + b_{rt} + l_{rt} && \forall r \in R,\ t \in T && \text{(backorder balance)} \\
& b_{rt} = \beta_r \left( b_{r,t-1} + d_{rt} - s_{rt} \right) && \forall r \in R,\ t < \tau && \text{(backorder share)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V \setminus \{w\},\ t \in T && \text{(storage capacity)} \\
& h_{wt} - u_t \le K^h_w && \forall t \in T && \text{(storage capacity of the central warehouse, relaxed by the rented storage)} \\
& u_t \le \bar u && \forall t \in T && \text{(rented storage)} \\
& p_{ft} - o_{ft} \le K_{ft}\, \delta_f && \forall f \in F,\ t \in T && \text{(assembly capacity, relaxed by the overtime)} \\
& o_{ft} \le \bar o_f\, \delta_f && \forall f \in F,\ t \in T && \text{(overtime)} \\
& b_{r\tau} = 0 \quad \forall r \in R, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required final stock at the end)} \\
& p_{ft},\ o_{ft},\ h_{it},\ u_t,\ x_{at},\ s_{rt},\ b_{rt},\ l_{rt} \ge 0, \qquad \delta_f \in \{0, 1\}
\end{aligned}
$$

with $h_{i0}$ given, $b_{r0} = 0$ and $w$ the central warehouse. The orders are $d_{rt} = \sigma_r\, D_t\, (1 + \gamma_t)$. What a plant assembles, $p_{ft}$, includes the bikes it assembles in overtime, and what the central warehouse holds includes the bikes held in rented storage. A closed plant assembles nothing: with $\delta_f = 0$ the assembly capacity and the overtime are 0. Plants 1 and 2 are always open.

## 3. Plan of a scenario

The cell solves M4B for the scenario built with the form and gives the cost and the margin of the plan, the margin under the base forecast at 700 €/bike with the same capacity options, the capacity utilization of each plant in the horizon, the price and the orders of each month, the bikes assembled at each plant with the part assembled in overtime, the seasonal stock at the end of each month by stage with the part held in rented storage, and the lost sales of each retailer. The figure gives the capacity utilization of each plant in each month, where the dashed line is the capacity, and the lost sales of each retailer. If several plans have the optimal cost, the cell says so.

**Demand growth.** `demand_growth` raises the orders of every retailer in every month by the same percentage over the base forecast. A negative value lowers them. Raise it from 0 to 40 %: from which growth on does the plan lose orders, and how much capacity has plant 3 left when that happens?

**Price promotion.** The tick boxes choose the months of the promotion, `discount` lowers the price of those months, and `growth_of_their_orders` raises their orders. The discount is paid on every order of those months. Put a discount of 7,5 % in April and May and raise the growth of their orders until the margin reaches the margin without promotion. Does the same discount in January to March need the same growth?

**Plant 3.** `plant_3` keeps plant 3 open, closes it for the six months, or leaves the decision to the model, which pays `fixed_cost_of_plant_3` for each month it is open. Close it under the base forecast: which retailers lose orders, and why those? With the decision left to the model and a fixed cost of 20.000 €/month, up to which demand growth does the model close it?

**Overtime and rented storage.** `overtime` lets plants 1 and 2 assemble above their capacity, at `overtime_cost` per bike above their unit assembly cost; `rented_storage` lets the central warehouse hold up to 200 bikes above its storage capacity, at `rented_storage_cost` per bike and month. With plant 3 closed, compare the two options one at a time and both together: which one delivers every order, and which one barely changes the plan?

In [ ]:
# @title Plan of a scenario { display-mode: "form" }
# Demand growth over the base forecast, in every month (%)
demand_growth = 0  # @param {type:"slider", min:-20, max:50, step:1}
# Months of the price promotion
january = False  # @param {type:"boolean"}
february = False  # @param {type:"boolean"}
march = False  # @param {type:"boolean"}
april = False  # @param {type:"boolean"}
may = False  # @param {type:"boolean"}
june = False  # @param {type:"boolean"}
# Discount on the price of those months (%) and growth of their orders (%)
discount = 0  # @param {type:"slider", min:0, max:30, step:0.5}
growth_of_their_orders = 0  # @param {type:"slider", min:0, max:150, step:1}
# Plant 3, and what it costs to keep it open (EUR/month)
plant_3 = "open"  # @param ["open", "closed", "model"]
fixed_cost_of_plant_3 = 0  # @param {type:"slider", min:0, max:50000, step:1000}
# Overtime at plants 1 and 2 (EUR/bike) and rented storage (EUR/(bike*month))
overtime = False  # @param {type:"boolean"}
overtime_cost = 100  # @param {type:"slider", min:0, max:200, step:10}
rented_storage = False  # @param {type:"boolean"}
rented_storage_cost = 12  # @param {type:"slider", min:0, max:40, step:1}
co.plan(growth=demand_growth,
        promotion_months=co.chosen_months(january, february, march, april, may, june),
        discount=discount, promotion_growth=growth_of_their_orders,
        plant_3=plant_3, fixed_cost=fixed_cost_of_plant_3,
        overtime=overtime, overtime_cost=overtime_cost,
        rented_storage=rented_storage, rented_cost=rented_storage_cost);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) explains all the numbers and has a script that prints them.